# Baseline Drowsiness Detection — PERCLOS & Rule-Based Fusion (RBF)

Subject-independent baseline models for the ES327 driver-drowsiness study.

This notebook extracts eye/mouth landmark features (EAR, MAR) with MediaPipe FaceLandmarker, computes PERCLOS and a rule-based fusion (RBF) drowsiness probability per clip, tunes the decision threshold on validation data, and evaluates on the UTA-RLDD, YawDD and DROZY test sets.

**Datasets:** UTA-RLDD, YawDD (internal) · DROZY (external)  
**Outputs:** `features_cache.csv`, `probs.csv`, `thresholds.json`, `results.json`

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip -q install numpy pandas scikit-learn tqdm mediapipe opencv-python

In [ ]:
!mkdir -p models
!wget -O models/face_landmarker.task \
https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

---
## 2. Baseline Pipeline (PERCLOS + RBF)

Self-contained pipeline: split loading → NPZ frame sampling → FaceLandmarker EAR/MAR features → PERCLOS & RBF probabilities → threshold tuning on validation → evaluation on all test sets.

In [ ]:
# ============================================================
# LARGE BASELINE PIPELINE (NPZ) — PERCLOS + RBF
# Uses: MediaPipe Tasks FaceLandmarker (Python 3.12 compatible)
# Loads splits from: ES327_Drowsiness/splits/*.csv
# ============================================================

import os, json, glob
import numpy as np
import pandas as pd
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, average_precision_score
)

# -----------------------------
# 0) DRIVE MOUNT (Colab)
# -----------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("[Info] Not in Colab / drive mount skipped:", e)

# -----------------------------
# 1) PATHS YOU ACTUALLY HAVE
# -----------------------------
BASE = "/content/drive/MyDrive/ES327_Drowsiness"
SPLITS_DIR = f"{BASE}/splits"
OUT_DIR = f"{BASE}/baseline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# FaceLandmarker model file
MODEL_PATH = f"{BASE}/models/face_landmarker.task"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Missing FaceLandmarker model at:\n{MODEL_PATH}\n\n"
        "Download once with:\n"
        f"!mkdir -p '{BASE}/models'\n"
        f"!wget -O '{MODEL_PATH}' "
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    )

print("[OK] BASE:", BASE)
print("[OK] SPLITS_DIR:", SPLITS_DIR)
print("[OK] MODEL_PATH:", MODEL_PATH)
print("[OK] OUT_DIR:", OUT_DIR)

# -----------------------------
# 2) CONFIG
# -----------------------------
N_FRAMES_MAIN = 4
N_FRAMES_DROZY = 2

EAR_THRESH = 0.21
MAR_THRESH = 0.65

W_PERCLOS = 0.60
W_YAWN   = 0.25
W_MARMAX = 0.15

MAR_NORM_DIV = 1.20  # deterministic scaling for mar_max -> [0,1]

# -----------------------------
# 3) LOAD ALL SPLITS FROM FILENAME
# expects filenames like yawdd_train.csv, uta_val.csv, drozy_test.csv
# -----------------------------
split_files = sorted(glob.glob(f"{SPLITS_DIR}/*.csv"))
if len(split_files) == 0:
    raise FileNotFoundError(f"No CSVs found in {SPLITS_DIR}")

dfs = []
for fp in split_files:
    name = os.path.basename(fp).lower().replace(".csv", "")
    # dataset_split (allow extra underscores in future)
    parts = name.split("_")
    if len(parts) < 2:
        print(f"[Skip] weird filename (need dataset_split): {fp}")
        continue
    dataset = parts[0].upper()   # YAWDD / UTA / DROZY
    split = parts[1].lower()     # train / val / test
    if split not in ("train", "val", "test"):
        print(f"[Skip] unknown split in filename: {fp}")
        continue

    d = pd.read_csv(fp)
    d["dataset"] = dataset
    d["split"] = split
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)

# ---------------------------------------
# OPTION C: Only compute baseline for val/test splits
# ---------------------------------------

df = df[
    ((df["dataset"].str.upper() == "YAWDD") & (df["split"].isin(["val", "test"]))) |
    ((df["dataset"].str.upper() == "UTA")   & (df["split"].isin(["val", "test"]))) |
    ((df["dataset"].str.upper() == "DROZY") & (df["split"] == "test"))
].copy()

print("\n[INFO] After dropping train splits:")
print(df.groupby(["dataset","split"]).size())

# ---- column normalization ----
# We need: clip_path, y_true, subject_id, dataset, split
# If your CSV uses different names, handle them here:

rename_map = {}
if "path" in df.columns and "clip_path" not in df.columns:
    rename_map["path"] = "clip_path"
if "label" in df.columns and "y_true" not in df.columns:
    rename_map["label"] = "y_true"
if "subject" in df.columns and "subject_id" not in df.columns:
    rename_map["subject"] = "subject_id"
if "subj" in df.columns and "subject_id" not in df.columns:
    rename_map["subj"] = "subject_id"

if rename_map:
    df = df.rename(columns=rename_map)

need = {"clip_path", "y_true", "subject_id", "dataset", "split"}
missing = need - set(df.columns)
if missing:
    raise ValueError(
        f"Your split CSVs are missing required columns: {missing}\n"
        f"Columns present: {list(df.columns)}\n\n"
        "Fix by renaming columns in the code (rename_map)."
    )

# Ensure types
df["clip_path"] = df["clip_path"].astype(str)
df["subject_id"] = df["subject_id"].astype(str)
df["y_true"] = df["y_true"].astype(int)
df["dataset"] = df["dataset"].astype(str)
df["split"] = df["split"].astype(str)

# If clip_path is relative, prefix BASE
def absolutize(p: str) -> str:
    p = p.strip()
    if p.startswith("/"):
        return p
    return f"{BASE}/{p}"

df["clip_path"] = df["clip_path"].apply(absolutize)

print("\n[Info] Clip counts by dataset/split:")
print(df.groupby(["dataset", "split"])["clip_path"].count())

# -----------------------------
# 4) NPZ LOADER + SAMPLER
# -----------------------------
def load_npz_frames(npz_path: str) -> np.ndarray:
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    data = np.load(npz_path, allow_pickle=False)
    if len(data.files) == 0:
        raise ValueError(f"NPZ has no arrays: {npz_path}")

    # prefer common keys
    if "frames" in data.files:
        arr = data["frames"]
    elif "clip" in data.files:
        arr = data["clip"]
    else:
        arr = data[data.files[0]]

    if arr.ndim != 4:
        raise ValueError(f"Expected 4D array, got {arr.shape} in {npz_path}")

    # Handle formats:
    # (T,H,W,C)
    if arr.shape[-1] in (1, 3) and arr.shape[0] != 3:
        thwc = arr

    # (T,C,H,W) -> (T,H,W,C)
    elif arr.shape[1] in (1, 3) and arr.shape[-1] not in (1, 3):
        thwc = np.transpose(arr, (0, 2, 3, 1))

    # (C,T,H,W) -> (T,H,W,C)  <-- your case
    elif arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
        thwc = np.transpose(arr, (1, 2, 3, 0))

    else:
        raise ValueError(f"Unrecognized 4D layout {arr.shape} in {npz_path}")

    # channels
    if thwc.shape[-1] == 1:
        thwc = np.repeat(thwc, 3, axis=-1)

    # dtype
    if thwc.dtype != np.uint8:
        mx = float(thwc.max())
        if mx <= 1.0:
            thwc = (thwc * 255.0).clip(0, 255).astype(np.uint8)
        else:
            thwc = thwc.clip(0, 255).astype(np.uint8)

    # IMPORTANT for mediapipe
    thwc = np.ascontiguousarray(thwc)

    return thwc  # (T,H,W,C) RGB uint8

def sample_even(frames: np.ndarray, n: int):
    T = frames.shape[0]
    if T <= 0 or n <= 0:
        return [], []
    idx = np.linspace(0, T - 1, n, dtype=int)
    idx = np.unique(idx)
    return [frames[i] for i in idx.tolist()], idx.tolist()

# -----------------------------
# 5) FaceLandmarker (Tasks API)
# -----------------------------
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)
landmarker = vision.FaceLandmarker.create_from_options(options)

def _pt(lms, idx: int):
    lm = lms[idx]
    return np.array([lm.x, lm.y], dtype=np.float32)

def _dist(a, b):
    return float(np.linalg.norm(a - b))

LEFT_EYE  = dict(p1=33,  p4=133, p2=160, p6=144, p3=158, p5=153)
RIGHT_EYE = dict(p1=362, p4=263, p2=385, p6=380, p3=387, p5=373)
MOUTH     = dict(left=61, right=291, upper=13, lower=14)

def compute_ear(lms):
    p1, p4 = _pt(lms, LEFT_EYE["p1"]), _pt(lms, LEFT_EYE["p4"])
    p2, p6 = _pt(lms, LEFT_EYE["p2"]), _pt(lms, LEFT_EYE["p6"])
    p3, p5 = _pt(lms, LEFT_EYE["p3"]), _pt(lms, LEFT_EYE["p5"])
    ear_l = (_dist(p2, p6) + _dist(p3, p5)) / (2.0 * max(_dist(p1, p4), 1e-6))

    p1, p4 = _pt(lms, RIGHT_EYE["p1"]), _pt(lms, RIGHT_EYE["p4"])
    p2, p6 = _pt(lms, RIGHT_EYE["p2"]), _pt(lms, RIGHT_EYE["p6"])
    p3, p5 = _pt(lms, RIGHT_EYE["p3"]), _pt(lms, RIGHT_EYE["p5"])
    ear_r = (_dist(p2, p6) + _dist(p3, p5)) / (2.0 * max(_dist(p1, p4), 1e-6))

    return float((ear_l + ear_r) / 2.0)

def compute_mar(lms):
    left  = _pt(lms, MOUTH["left"])
    right = _pt(lms, MOUTH["right"])
    upper = _pt(lms, MOUTH["upper"])
    lower = _pt(lms, MOUTH["lower"])
    w = max(_dist(left, right), 1e-6)
    h = _dist(upper, lower)
    return float(h / w)

def extract_feats(frame_rgb_uint8):
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb_uint8)
    res = landmarker.detect(mp_img)
    if not res.face_landmarks:
        return None
    lms = res.face_landmarks[0]
    return {"ear": compute_ear(lms), "mar": compute_mar(lms)}

# -----------------------------
# 6) Feature cache (one pass)
# -----------------------------
feature_rows = []
load_fail = 0

for _, r in tqdm(df.iterrows(), total=len(df), desc="(1) Cache features"):
    clip_path = r["clip_path"]
    dataset   = r["dataset"]
    split     = r["split"]
    subject   = r["subject_id"]
    y_true    = int(r["y_true"])

    n_frames = N_FRAMES_DROZY if dataset.lower() == "drozy" else N_FRAMES_MAIN

    try:
        frames = load_npz_frames(clip_path)
        sampled_frames, sampled_idx = sample_even(frames, n_frames)
    except Exception as e:
        load_fail += 1
        feature_rows.append({
            "clip_path": clip_path, "dataset": dataset, "split": split, "subject_id": subject, "y_true": y_true,
            "valid_frames": 0, "closed_frames": 0, "yawn_frames": 0,
            "perclos": 0.0, "yawn_prop": 0.0, "mar_max": 0.0,
            "sampled_idx": json.dumps([]),
            "error": f"LOAD_FAIL: {type(e).__name__}: {e}"
        })
        continue

    valid = closed = yawn = 0
    mar_max = 0.0

    for fr in sampled_frames:
        feats = extract_feats(fr)
        if feats is None:
            continue

        valid += 1
        ear = feats["ear"]
        mar = feats["mar"]

        if ear < EAR_THRESH:
            closed += 1
        if mar > MAR_THRESH:
            yawn += 1
        if mar > mar_max:
            mar_max = mar

    perclos   = (closed / valid) if valid > 0 else 0.0
    yawn_prop = (yawn   / valid) if valid > 0 else 0.0

    feature_rows.append({
        "clip_path": clip_path, "dataset": dataset, "split": split, "subject_id": subject, "y_true": y_true,
        "valid_frames": int(valid),
        "closed_frames": int(closed),
        "yawn_frames": int(yawn),
        "perclos": float(perclos),
        "yawn_prop": float(yawn_prop),
        "mar_max": float(mar_max),
        "sampled_idx": json.dumps(sampled_idx),
        "error": ""
    })

df_feat = pd.DataFrame(feature_rows)
feat_path = f"{OUT_DIR}/features_cache.csv"
df_feat.to_csv(feat_path, index=False)

print("\n[OK] saved:", feat_path)
print("[Info] load_fail:", load_fail)
print("[Info] valid_frames==0 rate:", float((df_feat["valid_frames"] == 0).mean()))

# -----------------------------
# 7) Probabilities
# -----------------------------
df_feat["norm_mar_max"] = (df_feat["mar_max"] / MAR_NORM_DIV).clip(0.0, 1.0)
df_feat["prob_perclos"] = df_feat["perclos"].astype(float)
df_feat["prob_rbf"] = (
    W_PERCLOS * df_feat["perclos"].astype(float)
    + W_YAWN   * df_feat["yawn_prop"].astype(float)
    + W_MARMAX * df_feat["norm_mar_max"].astype(float)
).clip(0.0, 1.0)

probs_path = f"{OUT_DIR}/probs.csv"
df_feat.to_csv(probs_path, index=False)
print("[OK] saved:", probs_path)

# -----------------------------
# 8) Metrics + threshold tuning (Option 3)
# -----------------------------
def compute_metrics(y_true, y_score, threshold: float):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        pr_auc = average_precision_score(y_true, y_score)
    except Exception:
        pr_auc = 0.0

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    spec = tn / (tn + fp) if (tn + fp) else 0.0
    fpr  = fp / (fp + tn) if (fp + tn) else 0.0
    fnr  = fn / (fn + tp) if (fn + tp) else 0.0

    return {
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision": float(prec),
        "recall_tpr": float(rec),
        "f1": float(f1),
        "pr_auc": float(pr_auc),
        "specificity_tnr": float(spec),
        "fpr": float(fpr),
        "fnr": float(fnr),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    }

def subset(df, dataset, split):
    return df[(df["dataset"].str.lower() == dataset.lower()) & (df["split"].str.lower() == split.lower())]

def best_threshold_f1(y_true, y_score, grid):
    best = None
    best_f1 = -1.0
    for t in grid:
        m = compute_metrics(y_true, y_score, float(t))
        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best = m
    return best

df_tune = df_feat[df_feat["valid_frames"] > 0].copy()
grid = np.linspace(0.0, 1.0, 101)

schemes = {
    "tune_on_yawdd_val": subset(df_tune, "YAWDD", "val"),
    "tune_on_uta_val": subset(df_tune, "UTA", "val"),
    "tune_on_combined_val": pd.concat([subset(df_tune, "YAWDD", "val"), subset(df_tune, "UTA", "val")], ignore_index=True),
}

thresholds = {}
for scheme_name, dval in schemes.items():
    if len(dval) == 0:
        thresholds[scheme_name] = {"error": "no validation rows"}
        continue

    thresholds[scheme_name] = {}
    for key in ["prob_perclos", "prob_rbf"]:
        best = best_threshold_f1(dval["y_true"], dval[key], grid)
        thresholds[scheme_name][key] = {"best_threshold": best["threshold"], "val_metrics": best}

thr_path = f"{OUT_DIR}/thresholds.json"
with open(thr_path, "w") as f:
    json.dump(thresholds, f, indent=2)
print("[OK] saved:", thr_path)

# -----------------------------
# 9) Evaluate on all test sets
# -----------------------------
tests = {
    "YAWDD_test": subset(df_feat, "YAWDD", "test"),
    "UTA_test": subset(df_feat, "UTA", "test"),
    "DROZY_test": subset(df_feat, "DROZY", "test"),
}

results = {}
for scheme_name, scheme in thresholds.items():
    if "error" in scheme:
        results[scheme_name] = scheme
        continue

    results[scheme_name] = {}
    for test_name, dtest in tests.items():
        results[scheme_name][test_name] = {}
        for key in ["prob_perclos", "prob_rbf"]:
            th = scheme[key]["best_threshold"]
            m = compute_metrics(dtest["y_true"], dtest[key], th)
            m["valid_frames_zero_rate"] = float((dtest["valid_frames"] == 0).mean())
            results[scheme_name][test_name][key] = m

    # robustness delta on RBF
    try:
        f1_uta = results[scheme_name]["UTA_test"]["prob_rbf"]["f1"]
        f1_dro = results[scheme_name]["DROZY_test"]["prob_rbf"]["f1"]
        results[scheme_name]["deltaF1_UTA_minus_DROZY_rbf"] = float(f1_uta - f1_dro)
    except Exception:
        pass

res_path = f"{OUT_DIR}/results.json"
with open(res_path, "w") as f:
    json.dump(results, f, indent=2)

print("[OK] saved:", res_path)
print("\n[Done] Baseline pipeline complete.")
print("Outputs:", OUT_DIR)

---
## 3. Results Summary

Reads `results.json` (combined-validation tuning scheme) into a per-dataset metrics table and reports the cross-domain robustness gap (ΔF1 between internal UTA-RLDD and external DROZY).

In [ ]:
import json, pandas as pd

BASE="/content/drive/MyDrive/ES327_Drowsiness"
OUT_DIR=f"{BASE}/baseline_outputs"

with open(f"{OUT_DIR}/results.json","r") as f:
    results = json.load(f)

scheme = results["tune_on_combined_val"]

rows=[]
for test_name in ["YAWDD_test","UTA_test","DROZY_test"]:
    for model_key in ["prob_perclos","prob_rbf"]:
        m = scheme[test_name][model_key]
        rows.append({
            "test_set": test_name,
            "baseline": model_key.replace("prob_","").upper(),
            "threshold": m["threshold"],
            "F1": m["f1"],
            "Accuracy": m["accuracy"],
            "Precision": m["precision"],
            "Recall": m["recall_tpr"],
            "PR-AUC": m["pr_auc"],
            "Specificity": m["specificity_tnr"],
            "valid_frames_zero_rate": m["valid_frames_zero_rate"],
            "TN": m["confusion_matrix"]["tn"],
            "FP": m["confusion_matrix"]["fp"],
            "FN": m["confusion_matrix"]["fn"],
            "TP": m["confusion_matrix"]["tp"],
        })

df_res = pd.DataFrame(rows)
df_res

print("\nΔF1 (UTA_test - DROZY_test) for RBF:",
      scheme["deltaF1_UTA_minus_DROZY_rbf"])